# Final Project: Duration-Aware Bank Marketing Optimization

This project studies how a bank can improve the efficiency of its term deposit marketing campaign using machine learning. The main goal is to predict which clients are likely to subscribe and then translate those predictions into a targeting decision under time and cost constraints. In addition to subscription probability, I also incorporate expected call duration as a cost dimension in the optimization stage.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

pd.set_option("display.max_columns", 100)
pd.set_option("display.width", 120)

## Load and Inspect the Dataset

I use the Bank Marketing dataset from the UCI Machine Learning Repository. Each row represents one client contact record from a Portuguese bank’s direct marketing campaign. The target variable is `y`, which indicates whether the client subscribed to a term deposit. I begin by loading the data, checking its structure, and reviewing the class balance and duration distribution.

In [ ]:
df = pd.read_csv("bank-additional-full.csv", sep=";")

print("Shape:", df.shape)
display(df.head())
display(df.info())

In [ ]:
print("Columns:")
print(df.columns.tolist())

print("\nTarget distribution:")
print(df["y"].value_counts())
print(df["y"].value_counts(normalize=True))

print("\nDuration summary:")
print(df["duration"].describe())

print("\nMissing values:")
display(df.isna().sum().sort_values(ascending=False))

## Business Framing and Variable Design
This project aims to support a bank marketing manager in deciding which clients to contact for a term deposit campaign.

A key modeling issue is that the observed call duration is only known after the call takes place, so it should not be used directly in a pre-call subscription prediction model. However, duration is economically important because longer calls consume more staff time and increase campaign cost. Therefore, I will exclude observed duration from the subscription classifier, but I will incorporate duration into the optimization stage by modeling expected call time as a cost dimension.

In [ ]:
target_col = "y"
duration_col = "duration"

# features available before the call decision
feature_cols = [c for c in df.columns if c not in [target_col, duration_col]]

print("Number of pre-call features:", len(feature_cols))
print(feature_cols)

In [ ]:
X = df[feature_cols].copy()
y = (df[target_col] == "yes").astype(int)

categorical_cols = X.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X.select_dtypes(exclude=["object"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

## Exploratory Data Analysis

I next conduct a small amount of exploratory analysis to better understand the target imbalance and the distribution of call duration. This is important because the positive class is relatively rare, so accuracy alone will not be enough for evaluation. I also examine duration because it will later be used as a cost dimension in the optimization stage.

In [ ]:
df["y"].value_counts().plot(kind="bar")
plt.title("Target Distribution")
plt.xlabel("Subscription")
plt.ylabel("Count")
plt.show()

In [ ]:
plt.hist(df["duration"], bins=50)
plt.title("Distribution of Call Duration")
plt.xlabel("Duration (seconds)")
plt.ylabel("Frequency")
plt.show()

In [ ]:
plt.hist(np.log1p(df["duration"]), bins=50)
plt.title("Log-Transformed Duration Distribution")
plt.xlabel("log(1 + duration)")
plt.ylabel("Frequency")
plt.show()

## Step 1: Baseline Pre-Call Subscription Prediction
In this section, I build a baseline classification model to predict whether a client will subscribe to a term deposit using only information available before the call. I exclude the observed call duration because it is only known after the call and would create data leakage in a pre-call decision setting.

In [ ]:
from sklearn.model_selection import train_test_split

# classification target
X_cls = df[feature_cols].copy()
y_cls = (df["y"] == "yes").astype(int)

# first split: train+valid vs test
X_train_valid, X_test, y_train_valid, y_test = train_test_split(
    X_cls, y_cls,
    test_size=0.2,
    random_state=42,
    stratify=y_cls
)

# second split: train vs valid
X_train, X_valid, y_train, y_valid = train_test_split(
    X_train_valid, y_train_valid,
    test_size=0.25,   # 0.25 of 0.8 = 0.2 total
    random_state=42,
    stratify=y_train_valid
)

print("Train shape:", X_train.shape)
print("Valid shape:", X_valid.shape)
print("Test shape:", X_test.shape)

print("\nTrain target rate:", y_train.mean())
print("Valid target rate:", y_valid.mean())
print("Test target rate:", y_test.mean())

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer

# numeric and categorical columns
categorical_cols = X_train.select_dtypes(include=["object"]).columns.tolist()
numeric_cols = X_train.select_dtypes(exclude=["object"]).columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_cols),
    ("cat", categorical_transformer, categorical_cols)
])

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

In [ ]:
from sklearn.linear_model import LogisticRegression

logit_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LogisticRegression(max_iter=2000, random_state=42))
])

logit_model.fit(X_train, y_train)

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, average_precision_score, confusion_matrix,
    ConfusionMatrixDisplay, RocCurveDisplay, PrecisionRecallDisplay,
    brier_score_loss
)
import matplotlib.pyplot as plt

def evaluate_classifier(model, X, y, split_name="Validation", threshold=0.5):
    y_prob = model.predict_proba(X)[:, 1]
    y_pred = (y_prob >= threshold).astype(int)

    metrics = {
        "split": split_name,
        "threshold": threshold,
        "accuracy": accuracy_score(y, y_pred),
        "precision": precision_score(y, y_pred, zero_division=0),
        "recall": recall_score(y, y_pred, zero_division=0),
        "f1": f1_score(y, y_pred, zero_division=0),
        "roc_auc": roc_auc_score(y, y_prob),
        "average_precision": average_precision_score(y, y_prob),
        "brier_score": brier_score_loss(y, y_prob)
    }

    print(pd.Series(metrics))

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))

    ConfusionMatrixDisplay.from_predictions(y, y_pred, ax=axes[0], colorbar=False)
    axes[0].set_title(f"{split_name} Confusion Matrix")

    RocCurveDisplay.from_predictions(y, y_prob, ax=axes[1])
    axes[1].set_title(f"{split_name} ROC Curve")

    PrecisionRecallDisplay.from_predictions(y, y_prob, ax=axes[2])
    axes[2].set_title(f"{split_name} Precision-Recall Curve")

    plt.tight_layout()
    plt.show()

    return metrics, y_prob

In [ ]:
valid_metrics_logit, valid_prob_logit = evaluate_classifier(
    logit_model, X_valid, y_valid, split_name="Validation", threshold=0.5
)

## Step 2: Stronger Nonlinear Classification Model
To improve on the logistic regression baseline, I train a random forest classifier. This model can capture nonlinear relationships and interactions between pre-call customer features, which may improve ranking quality for marketing targeting.

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(
        n_estimators=300,
        max_depth=None,
        min_samples_leaf=5,
        class_weight="balanced_subsample",
        random_state=42,
        n_jobs=-1
    ))
])

rf_model.fit(X_train, y_train)

In [ ]:
valid_metrics_rf, valid_prob_rf = evaluate_classifier(
    rf_model, X_valid, y_valid, split_name="Validation RF", threshold=0.5
)

In [ ]:
comparison_df = pd.DataFrame([
    valid_metrics_logit,
    valid_metrics_rf
])

display(comparison_df)

## Step 3: Probability Calibration
Because the downstream business optimization uses predicted probabilities in expected profit calculations, probability calibration matters in addition to classification accuracy or ranking quality. I therefore calibrate the random forest model and compare its probability quality with the logistic regression baseline.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

rf_calibrated = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", CalibratedClassifierCV(
        estimator=RandomForestClassifier(
            n_estimators=300,
            max_depth=None,
            min_samples_leaf=5,
            class_weight="balanced_subsample",
            random_state=42,
            n_jobs=-1
        ),
        method="sigmoid",
        cv=3
    ))
])

rf_calibrated.fit(X_train, y_train)

In [ ]:
valid_metrics_rf_cal, valid_prob_rf_cal = evaluate_classifier(
    rf_calibrated, X_valid, y_valid, split_name="Validation RF Calibrated", threshold=0.5
)

In [ ]:
comparison_df = pd.DataFrame([
    valid_metrics_logit,
    valid_metrics_rf,
    valid_metrics_rf_cal
])

display(comparison_df)

### Interpretation of Classification Results

The calibrated random forest provides the best overall balance of ranking quality and probability quality. It slightly improves ROC-AUC and average precision relative to the baseline models, while also achieving the best Brier score. Since the downstream business optimization uses predicted probabilities in expected-profit calculations, I use the calibrated random forest as the final subscription model.

In [ ]:
from sklearn.calibration import calibration_curve

def plot_calibration_curves(y_true, prob_dict, n_bins=10):
    plt.figure(figsize=(7, 6))

    for name, probs in prob_dict.items():
        frac_pos, mean_pred = calibration_curve(y_true, probs, n_bins=n_bins, strategy="quantile")
        plt.plot(mean_pred, frac_pos, marker="o", label=name)

    plt.plot([0, 1], [0, 1], linestyle="--", color="gray", label="Perfect calibration")
    plt.xlabel("Mean predicted probability")
    plt.ylabel("Observed fraction of positives")
    plt.title("Calibration Curve")
    plt.legend()
    plt.show()

plot_calibration_curves(
    y_valid,
    {
        "Logistic Regression": valid_prob_logit,
        "Random Forest": valid_prob_rf,
        "Calibrated RF": valid_prob_rf_cal
    }
)

## Step 4: Predicting Expected Call Duration
Following the instructor feedback, I incorporate call duration into the business optimization as a cost dimension. However, because observed duration is only known after the call, I do not use it directly in the subscription classifier. Instead, I build a separate regression model to predict expected call duration from pre-call features, and I use the predicted duration in the optimization stage.

In [ ]:
# duration target
y_dur = df["duration"].copy()

# use the same row splits as the classification task
y_dur_train = y_dur.loc[X_train.index]
y_dur_valid = y_dur.loc[X_valid.index]
y_dur_test = y_dur.loc[X_test.index]

print("Train duration shape:", y_dur_train.shape)
print("Valid duration shape:", y_dur_valid.shape)
print("Test duration shape:", y_dur_test.shape)

print("\nTrain duration summary:")
print(y_dur_train.describe())

In [ ]:
y_dur_train_log = np.log1p(y_dur_train)
y_dur_valid_log = np.log1p(y_dur_valid)
y_dur_test_log = np.log1p(y_dur_test)

In [ ]:
from sklearn.linear_model import LinearRegression

dur_lr_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

dur_lr_model.fit(X_train, y_dur_train_log)

In [ ]:
from sklearn.ensemble import RandomForestRegressor

dur_rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(
        n_estimators=300,
        min_samples_leaf=10,
        random_state=42,
        n_jobs=-1
    ))
])

dur_rf_model.fit(X_train, y_dur_train_log)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

def evaluate_duration_model(model, X, y_true, split_name="Validation"):
    # predict on log scale
    y_pred_log = model.predict(X)

    # back-transform to original seconds scale
    y_pred = np.expm1(y_pred_log)
    y_pred = np.clip(y_pred, 0, None)

    metrics = {
        "split": split_name,
        "mae_seconds": mean_absolute_error(y_true, y_pred),
        "rmse_seconds": np.sqrt(mean_squared_error(y_true, y_pred)),
        "r2": r2_score(y_true, y_pred)
    }

    print(pd.Series(metrics))

    # quick scatter plot using a sample for readability
    sample_idx = np.random.RandomState(42).choice(len(y_true), size=min(1000, len(y_true)), replace=False)

    plt.figure(figsize=(6, 6))
    plt.scatter(np.array(y_true)[sample_idx], np.array(y_pred)[sample_idx], alpha=0.3)
    max_val = max(np.max(np.array(y_true)[sample_idx]), np.max(np.array(y_pred)[sample_idx]))
    plt.plot([0, max_val], [0, max_val], linestyle="--", color="red")
    plt.xlabel("Actual duration (seconds)")
    plt.ylabel("Predicted duration (seconds)")
    plt.title(f"{split_name}: Actual vs Predicted Duration")
    plt.show()

    return metrics, y_pred

In [ ]:
valid_metrics_dur_lr, valid_pred_dur_lr = evaluate_duration_model(
    dur_lr_model, X_valid, y_dur_valid, split_name="Validation Duration LR"
)

valid_metrics_dur_rf, valid_pred_dur_rf = evaluate_duration_model(
    dur_rf_model, X_valid, y_dur_valid, split_name="Validation Duration RF"
)

In [ ]:
duration_comparison_df = pd.DataFrame([
    valid_metrics_dur_lr,
    valid_metrics_dur_rf
])

display(duration_comparison_df)

### Interpretation of Duration Results

Expected call duration is much harder to predict than subscription probability. Both regression models have limited predictive power, which suggests that call length depends on many factors that are not known before the call. For this reason, the duration model should be interpreted as an approximate cost proxy rather than a precise forecast. I still use the random forest regressor because it performs slightly better than the linear baseline.

## Step 5: Duration-Aware Business Optimization

I now combine the calibrated subscription model and the duration prediction model to support a marketing targeting decision. The bank should not contact every client, because calls consume time and cost money. Instead, I estimate the expected profit of contacting each client using both predicted subscription probability and predicted call duration, and compare this duration-aware policy with simpler baseline strategies.

In [ ]:
# business assumptions
R = 100.0              # revenue from a successful subscription
c0 = 1.0               # fixed cost per call
c1_per_min = 4.0       # variable cost per minute of call time
budget_minutes = 3000  # total outbound call-time budget

print("Revenue per subscription:", R)
print("Fixed call cost:", c0)
print("Variable cost per minute:", c1_per_min)
print("Budget (minutes):", budget_minutes)

In [ ]:
decision_valid = pd.DataFrame({
    "p_subscribe": valid_prob_rf_cal,          # calibrated RF predicted probability
    "pred_duration_sec": valid_pred_dur_rf,    # RF predicted duration in seconds
    "actual_y": y_valid.values,                # actual subscription outcome
    "actual_duration_sec": y_dur_valid.values  # actual duration in seconds
}, index=X_valid.index)

decision_valid["pred_duration_min"] = decision_valid["pred_duration_sec"] / 60.0
decision_valid["actual_duration_min"] = decision_valid["actual_duration_sec"] / 60.0

decision_valid["expected_profit"] = (
    decision_valid["p_subscribe"] * R
    - c0
    - c1_per_min * decision_valid["pred_duration_min"]
)

decision_valid["realized_profit"] = (
    decision_valid["actual_y"] * R
    - c0
    - c1_per_min * decision_valid["actual_duration_min"]
)

decision_valid["expected_profit_per_min"] = (
    decision_valid["expected_profit"] / decision_valid["pred_duration_min"].clip(lower=1e-6)
)

decision_valid.head()

In [ ]:
print(decision_valid["expected_profit"].describe())

plt.figure(figsize=(7, 5))
plt.hist(decision_valid["expected_profit"], bins=50)
plt.title("Distribution of Expected Profit per Client")
plt.xlabel("Expected Profit")
plt.ylabel("Count")
plt.show()

In [ ]:
def select_under_budget(df, score_col, budget_minutes, duration_col="pred_duration_min",
                        ascending=False, require_positive_profit=False):
    temp = df.copy()

    if require_positive_profit:
        temp = temp[temp["expected_profit"] > 0].copy()

    temp = temp.sort_values(score_col, ascending=ascending)
    temp["cum_minutes"] = temp[duration_col].cumsum()

    selected = temp[temp["cum_minutes"] <= budget_minutes].copy()
    return selected.index

In [ ]:
rng = np.random.RandomState(42)
decision_valid["random_score"] = rng.rand(len(decision_valid))

# 1. contact nobody
idx_none = decision_valid.iloc[[]].index

# 2. contact everyone (reference only; may exceed budget)
idx_all = decision_valid.index

# 3. random targeting under budget
idx_random = select_under_budget(
    decision_valid,
    score_col="random_score",
    budget_minutes=budget_minutes,
    ascending=False
)

# 4. probability-only targeting under budget
idx_prob = select_under_budget(
    decision_valid,
    score_col="p_subscribe",
    budget_minutes=budget_minutes,
    ascending=False
)

# 5. duration-aware targeting under budget
idx_duration_aware = select_under_budget(
    decision_valid,
    score_col="expected_profit_per_min",
    budget_minutes=budget_minutes,
    ascending=False,
    require_positive_profit=True
)

In [ ]:
def summarize_policy(df, selected_idx, policy_name, budget_minutes=None):
    sel = df.loc[selected_idx].copy()

    pred_minutes = sel["pred_duration_min"].sum()
    actual_minutes = sel["actual_duration_min"].sum()
    conversions = int(sel["actual_y"].sum())
    expected_profit = sel["expected_profit"].sum()
    realized_profit = sel["realized_profit"].sum()

    summary = {
        "policy": policy_name,
        "n_contacted": len(sel),
        "pred_total_minutes": pred_minutes,
        "actual_total_minutes": actual_minutes,
        "within_budget": (pred_minutes <= budget_minutes) if budget_minutes is not None else True,
        "conversions": conversions,
        "expected_total_profit": expected_profit,
        "realized_total_profit": realized_profit,
        "realized_profit_per_hour": (
            realized_profit / (actual_minutes / 60.0) if actual_minutes > 0 else 0.0
        )
    }
    return summary

In [ ]:
policy_results_valid = pd.DataFrame([
    summarize_policy(decision_valid, idx_none, "Contact Nobody", budget_minutes),
    summarize_policy(decision_valid, idx_all, "Contact Everyone", budget_minutes),
    summarize_policy(decision_valid, idx_random, "Random Under Budget", budget_minutes),
    summarize_policy(decision_valid, idx_prob, "Top Probability Under Budget", budget_minutes),
    summarize_policy(decision_valid, idx_duration_aware, "Duration-Aware Under Budget", budget_minutes),
])

display(policy_results_valid)

In [ ]:
plt.figure(figsize=(10, 5))
plt.bar(policy_results_valid["policy"], policy_results_valid["realized_total_profit"])
plt.xticks(rotation=30, ha="right")
plt.title("Validation Realized Profit by Policy")
plt.ylabel("Realized Profit")
plt.show()

## Step 6: Sensitivity Analysis
The relative performance of different targeting policies may depend on the assumed revenue per successful subscription, the cost of call time, and the overall call-time budget. I therefore conduct a sensitivity analysis to examine when a duration-aware policy becomes more valuable than a probability-only targeting policy.

In [ ]:
def run_policy_comparison(df, R, c0, c1_per_min, budget_minutes, random_state=42):
    temp = df.copy()

    temp["expected_profit"] = (
        temp["p_subscribe"] * R
        - c0
        - c1_per_min * temp["pred_duration_min"]
    )

    temp["realized_profit"] = (
        temp["actual_y"] * R
        - c0
        - c1_per_min * temp["actual_duration_min"]
    )

    temp["expected_profit_per_min"] = (
        temp["expected_profit"] / temp["pred_duration_min"].clip(lower=1e-6)
    )

    rng = np.random.RandomState(random_state)
    temp["random_score"] = rng.rand(len(temp))

    idx_none = temp.iloc[[]].index
    idx_all = temp.index

    idx_random = select_under_budget(
        temp,
        score_col="random_score",
        budget_minutes=budget_minutes,
        ascending=False
    )

    idx_prob = select_under_budget(
        temp,
        score_col="p_subscribe",
        budget_minutes=budget_minutes,
        ascending=False
    )

    idx_duration = select_under_budget(
        temp,
        score_col="expected_profit_per_min",
        budget_minutes=budget_minutes,
        ascending=False,
        require_positive_profit=True
    )

    results = pd.DataFrame([
        summarize_policy(temp, idx_none, "Contact Nobody", budget_minutes),
        summarize_policy(temp, idx_all, "Contact Everyone", budget_minutes),
        summarize_policy(temp, idx_random, "Random Under Budget", budget_minutes),
        summarize_policy(temp, idx_prob, "Top Probability Under Budget", budget_minutes),
        summarize_policy(temp, idx_duration, "Duration-Aware Under Budget", budget_minutes),
    ])

    results["R"] = R
    results["c0"] = c0
    results["c1_per_min"] = c1_per_min
    results["budget_minutes"] = budget_minutes

    return results

In [ ]:
results_list = []

for c1 in [1.0, 2.0, 4.0, 6.0, 8.0]:
    for B in [1500, 2500, 3000, 4000, 5000]:
        res = run_policy_comparison(
            decision_valid,
            R=100.0,
            c0=1.0,
            c1_per_min=c1,
            budget_minutes=B,
            random_state=42
        )
        results_list.append(res)

sensitivity_results = pd.concat(results_list, ignore_index=True)
display(sensitivity_results.head())

In [ ]:
policy_focus = sensitivity_results[
    sensitivity_results["policy"].isin([
        "Top Probability Under Budget",
        "Duration-Aware Under Budget"
    ])
].copy()

display(policy_focus.head(10))

In [ ]:
pivot_profit = policy_focus.pivot_table(
    index=["c1_per_min", "budget_minutes"],
    columns="policy",
    values="realized_total_profit"
).reset_index()

pivot_profit["duration_minus_prob"] = (
    pivot_profit["Duration-Aware Under Budget"]
    - pivot_profit["Top Probability Under Budget"]
)

display(pivot_profit)

In [ ]:
plt.figure(figsize=(9, 6))

for c1 in sorted(pivot_profit["c1_per_min"].unique()):
    sub = pivot_profit[pivot_profit["c1_per_min"] == c1]
    plt.plot(
        sub["budget_minutes"],
        sub["duration_minus_prob"],
        marker="o",
        label=f"c1_per_min = {c1}"
    )

plt.axhline(0, linestyle="--", color="gray")
plt.xlabel("Budget Minutes")
plt.ylabel("Duration-Aware Profit - Top Probability Profit")
plt.title("When Does Duration-Aware Targeting Outperform?")
plt.legend()
plt.show()

### Interpretation of Policy Sensitivity

The sensitivity analysis shows that the optimal targeting rule depends on operational assumptions rather than being universal. In the baseline setting, probability-only targeting tends to generate the highest realized profit. However, when call-center time is tighter or when time cost becomes more important, duration-aware targeting becomes more competitive and can outperform probability-only targeting. This means that call duration is not always the dominant factor, but it becomes valuable in more capacity-constrained environments.

## Step 7: Budget Overrun Risk and Safety Buffer
Because call duration is predicted rather than known in advance, a policy that satisfies the predicted budget may still exceed the actual realized call-time budget. To reduce this implementation risk, I evaluate a conservative targeting rule that only uses a fraction of the nominal budget during customer selection.

In [ ]:
def run_policy_comparison_buffered(df, R, c0, c1_per_min, budget_minutes,
                                   buffer_frac=1.0, random_state=42):
    temp = df.copy()

    temp["expected_profit"] = (
        temp["p_subscribe"] * R
        - c0
        - c1_per_min * temp["pred_duration_min"]
    )

    temp["realized_profit"] = (
        temp["actual_y"] * R
        - c0
        - c1_per_min * temp["actual_duration_min"]
    )

    temp["expected_profit_per_min"] = (
        temp["expected_profit"] / temp["pred_duration_min"].clip(lower=1e-6)
    )

    rng = np.random.RandomState(random_state)
    temp["random_score"] = rng.rand(len(temp))

    selection_budget = budget_minutes * buffer_frac

    idx_none = temp.iloc[[]].index
    idx_all = temp.index

    idx_random = select_under_budget(
        temp,
        score_col="random_score",
        budget_minutes=selection_budget,
        ascending=False
    )

    idx_prob = select_under_budget(
        temp,
        score_col="p_subscribe",
        budget_minutes=selection_budget,
        ascending=False
    )

    idx_duration = select_under_budget(
        temp,
        score_col="expected_profit_per_min",
        budget_minutes=selection_budget,
        ascending=False,
        require_positive_profit=True
    )

    results = pd.DataFrame([
        summarize_policy(temp, idx_none, "Contact Nobody", budget_minutes),
        summarize_policy(temp, idx_all, "Contact Everyone", budget_minutes),
        summarize_policy(temp, idx_random, "Random Under Budget", budget_minutes),
        summarize_policy(temp, idx_prob, "Top Probability Under Budget", budget_minutes),
        summarize_policy(temp, idx_duration, "Duration-Aware Under Budget", budget_minutes),
    ])

    results["R"] = R
    results["c0"] = c0
    results["c1_per_min"] = c1_per_min
    results["budget_minutes"] = budget_minutes
    results["buffer_frac"] = buffer_frac
    results["selection_budget_minutes"] = selection_budget

    return results

In [ ]:
base_unbuffered = run_policy_comparison_buffered(
    decision_valid, R=100, c0=1, c1_per_min=4, budget_minutes=3000, buffer_frac=1.0
)

base_buffer_85 = run_policy_comparison_buffered(
    decision_valid, R=100, c0=1, c1_per_min=4, budget_minutes=3000, buffer_frac=0.85
)

base_buffer_70 = run_policy_comparison_buffered(
    decision_valid, R=100, c0=1, c1_per_min=4, budget_minutes=3000, buffer_frac=0.70
)

buffer_compare = pd.concat(
    [base_unbuffered, base_buffer_85, base_buffer_70],
    ignore_index=True
)

display(buffer_compare)

In [ ]:
buffer_focus = buffer_compare[
    buffer_compare["policy"].isin([
        "Top Probability Under Budget",
        "Duration-Aware Under Budget"
    ])
].copy()

display(buffer_focus)

## Step 8: Final Evaluation on the Test Set
After selecting the main models and business scenarios using the validation set, I now evaluate the final targeting policies on the held-out test set. The test set is used only for final reporting and not for model selection or policy tuning.

In [ ]:
# test predictions from final selected models

p_test = rf_calibrated.predict_proba(X_test)[:, 1]

pred_dur_test = np.expm1(dur_rf_model.predict(X_test))
pred_dur_test = np.clip(pred_dur_test, 0, None)

decision_test = pd.DataFrame({
    "p_subscribe": p_test,
    "pred_duration_sec": pred_dur_test,
    "actual_y": y_test.values,
    "actual_duration_sec": y_dur_test.values
}, index=X_test.index)

decision_test["pred_duration_min"] = decision_test["pred_duration_sec"] / 60.0
decision_test["actual_duration_min"] = decision_test["actual_duration_sec"] / 60.0

decision_test.head()

In [ ]:
test_base_results = run_policy_comparison_buffered(
    decision_test,
    R=100,
    c0=1,
    c1_per_min=4,
    budget_minutes=3000,
    buffer_frac=1.0,
    random_state=42
)

display(test_base_results)

In [ ]:
def add_overrun_metrics(results_df):
    out = results_df.copy()
    out["actual_minus_budget"] = out["actual_total_minutes"] - out["budget_minutes"]
    out["actual_overrun_pct"] = out["actual_minus_budget"] / out["budget_minutes"]
    return out

test_base_results_overrun = add_overrun_metrics(test_base_results)
display(test_base_results_overrun)

### Interpretation of Budget Overrun Risk

Even when a targeting policy satisfies the predicted call-time budget, the realized call time may still exceed the budget because duration must be predicted rather than known in advance. In the baseline test scenario, both probability-only and duration-aware policies exceed the realized budget, which highlights the operational risk of relying on noisy duration forecasts. This suggests that practical deployment may require a safety buffer.

In [ ]:
test_stress_results = run_policy_comparison_buffered(
    decision_test,
    R=100,
    c0=1,
    c1_per_min=1,
    budget_minutes=1500,
    buffer_frac=1.0,
    random_state=42
)

display(test_stress_results)

In [ ]:
test_stress_results_overrun = add_overrun_metrics(test_stress_results)
display(test_stress_results_overrun)

### Interpretation of Stress Scenario

In the tighter-budget stress scenario, duration-aware targeting produces higher realized profit than probability-only targeting. This suggests that expected call duration becomes more important when call-center capacity is more constrained.

## Notebook Summary

In this project, I studied how a bank can improve a term deposit marketing campaign using machine learning and business optimization. I used a calibrated random forest to predict subscription probability and a separate random forest regressor to approximate expected call duration from pre-call features. On the held-out test set, model-based targeting clearly outperformed naive strategies such as contacting everyone or selecting clients randomly. In the baseline business scenario, targeting clients by predicted subscription probability produced the highest realized profit. However, under a tighter call-time budget, duration-aware targeting became more profitable, showing that call duration can matter when operational capacity is constrained. Overall, the results suggest that the bank should use probability-only targeting in normal conditions, but switch to a duration-aware policy when staff time becomes scarcer or more operationally costly.